In [1]:
library(SPARK)
library(Seurat)
library(anndata)
library(reticulate)
use_condaenv(condaenv = "/home/bingxing2/ailab/scxlab0179/.conda/envs/bio/",required = TRUE)

Warning message:
“package ‘Seurat’ was built under R version 4.4.3”
Loading required package: SeuratObject

Warning message:
“package ‘SeuratObject’ was built under R version 4.4.3”
Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t



Attaching package: ‘anndata’


The following object is masked from ‘package:SeuratObject’:

    Layers




In [26]:
data <- read_h5ad("/home/bingxing2/ailab/group/ai4bio/sunjianle/mouse_pmc/rna_imputed.h5ad")
seurat_obj <- CreateSeuratObject(counts = t(data$X), meta.data = data$obs)
seurat_obj

Warning message:
“Data is of class dgRMatrix. Coercing to dgCMatrix.”


An object of class Seurat 
27123 features across 14107 samples within 1 assay 
Active assay: RNA (27123 features, 0 variable features)
 1 layer present: counts

In [27]:
seurat_obj <- NormalizeData(seurat_obj, normalization.method = "LogNormalize", scale.factor = 10000)
seurat_obj <- FindVariableFeatures(seurat_obj, selection.method = "vst", nfeatures = 3000)
var_genes <- VariableFeatures(seurat_obj)

Normalizing layer: counts

Finding variable features for layer counts



In [28]:
sp_counts <- seurat_obj$RNA$counts #[var_genes,]
location <- data$obsm$spatial

In [29]:
idx <- which(complete.cases(location))
cells_to_keep <- rownames(seurat_obj@meta.data)[idx]
seurat_obj <- subset(seurat_obj, cells = cells_to_keep)
sp_counts <- seurat_obj$RNA$counts
location <- location[idx,]
seurat_obj

An object of class Seurat 
27123 features across 13838 samples within 1 assay 
Active assay: RNA (27123 features, 3000 variable features)
 2 layers present: counts, data

In [30]:
sparkX <- sparkx(sp_counts,location,numCores=5,option="mixture")

## ===== SPARK-X INPUT INFORMATION ==== 
## number of total samples: 13838 
## number of total genes: 26038 
## Running with 5 cores 
## Testing With Projection Kernel
## Testing With Gaussian Kernel 1
## Testing With Gaussian Kernel 2
## Testing With Gaussian Kernel 3
## Testing With Gaussian Kernel 4
## Testing With Gaussian Kernel 5
## Testing With Cosine Kernel 1
## Testing With Cosine Kernel 2
## Testing With Cosine Kernel 3
## Testing With Cosine Kernel 4
## Testing With Cosine Kernel 5


Warning message in FUN(newX[, i], ...):
“There are p-values that are exactly 1!”


In [31]:
head(sparkX$res_mtest)

,combinedPval,adjustedPval
,<dbl>,<dbl>
Xkr4,3.899646e-118,1.179449e-115
Gm1992,1.434365e-30,1.233971e-28
Gm37381,6.020032e-01,1.000000e+00
Rp1,9.128706e-01,1.000000e+00
Sox17,5.933517e-01,1.000000e+00
Gm37323,9.363557e-01,1.000000e+00


In [32]:
library(tidyverse)
df <- sparkX$res_mtest %>% arrange(adjustedPval,combinedPval)
df$gene_name <- rownames(df)
df <- df %>% left_join(data$var, by="gene_name")
df |> head()

,combinedPval,adjustedPval,gene_name,gene_ids,feature_types,genome,chrom,chromStart,chromEnd,name,⋯,gene_type,mgi_id,havana_gene,tag,n_counts,highly_variable,highly_variable_rank,means,variances,variances_norm
,<dbl>,<dbl>,<chr>,<chr>,<fct>,<fct>,<fct>,<dbl>,<dbl>,<chr>,⋯,<fct>,<fct>,<fct>,<fct>,<dbl>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>
1,0,0,3110035E14Rik,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
2,0,0,Cpa6,ENSMUSG00000042501,Gene Expression,mm10,chr1,10324719,10719945,ENSMUSG00000042501,⋯,protein_coding,MGI:3045348,OTTMUSG00000024232.1,overlapping_locus,165124,TRUE,114,2.368150,53.62438,8.912715
3,0,0,A830018L16Rik,ENSMUSG00000057715,Gene Expression,mm10,chr1,11414104,11975901,ENSMUSG00000057715,⋯,protein_coding,MGI:2444149,OTTMUSG00000033915.2,overlapping_locus,1504514,TRUE,1273,21.577208,495.30218,1.970123
4,0,0,Sulf1,ENSMUSG00000016918,Gene Expression,mm10,chr1,12692276,12861192,ENSMUSG00000016918,⋯,protein_coding,MGI:2138563,OTTMUSG00000047995.3,NA,219512,TRUE,162,3.148164,63.55406,7.049831
5,0,0,Kcnq5,ENSMUSG00000028033,Gene Expression,mm10,chr1,21398402,21961942,ENSMUSG00000028033,⋯,protein_coding,MGI:1924937,OTTMUSG00000019675.5,overlapping_locus,5522624,TRUE,1885,79.203522,5181.52616,1.596328
6,0,0,Ogfrl1,ENSMUSG00000026158,Gene Expression,mm10,chr1,23366423,23397772,ENSMUSG00000026158,⋯,protein_coding,MGI:1917405,OTTMUSG00000046267.3,NA,793479,FALSE,NaN,11.379795,105.19379,1.374914


In [34]:
df %>% write_csv("/home/bingxing2/ailab/group/ai4bio/sunjianle/mouse_pmc/svg_rna.csv")

In [79]:
data <- read_h5ad("/home/bingxing2/ailab/group/ai4bio/sunjianle/mouse_pmc/st_processed.h5ad")
seurat_obj <- CreateSeuratObject(counts = t(data$layers[["counts"]]), meta.data = data$obs)
sp_counts <- seurat_obj$RNA$counts
location <- data$obsm$X_spatial
seurat_obj

Warning message:
“Data is of class dgRMatrix. Coercing to dgCMatrix.”


An object of class Seurat 
254 features across 6319 samples within 1 assay 
Active assay: RNA (254 features, 0 variable features)
 1 layer present: counts

In [80]:
sparkX2 <- sparkx(sp_counts,location,numCores=5,option="mixture")

## ===== SPARK-X INPUT INFORMATION ==== 
## number of total samples: 6319 
## number of total genes: 254 
## Running with 5 cores 
## Testing With Projection Kernel
## Testing With Gaussian Kernel 1
## Testing With Gaussian Kernel 2
## Testing With Gaussian Kernel 3
## Testing With Gaussian Kernel 4
## Testing With Gaussian Kernel 5
## Testing With Cosine Kernel 1
## Testing With Cosine Kernel 2
## Testing With Cosine Kernel 3
## Testing With Cosine Kernel 4
## Testing With Cosine Kernel 5


In [81]:
library(tidyverse)
df <- sparkX2$res_mtest %>% arrange(adjustedPval,combinedPval)
df$gene_name <- rownames(df)
data$var$gene_name <- rownames(data$var)
df <- df %>% left_join(data$var, by="gene_name")
df %>% write_csv("/home/bingxing2/ailab/group/ai4bio/sunjianle/mouse_pmc/svg_st.csv")

In [20]:
data <- read_h5ad("/home/bingxing2/ailab/group/ai4bio/sunjianle/mouse_pmc/atac_gas_imputed.h5ad")
seurat_obj <- CreateSeuratObject(counts = t(data$X), meta.data = data$obs)
sp_counts <- seurat_obj$RNA$counts
location <- data$obsm$spatial
seurat_obj

Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”


An object of class Seurat 
20563 features across 7102 samples within 1 assay 
Active assay: RNA (20563 features, 0 variable features)
 1 layer present: counts

In [21]:
idx <- which(complete.cases(location))
cells_to_keep <- rownames(seurat_obj@meta.data)[idx]
seurat_obj <- subset(seurat_obj, cells = cells_to_keep)
sp_counts <- seurat_obj$RNA$counts
location <- location[idx,]
seurat_obj

An object of class Seurat 
20563 features across 7051 samples within 1 assay 
Active assay: RNA (20563 features, 0 variable features)
 1 layer present: counts

In [22]:
sparkX3 <- sparkx(sp_counts,location,numCores=5,option="mixture")

## ===== SPARK-X INPUT INFORMATION ==== 
## number of total samples: 7051 
## number of total genes: 20263 
## Running with 5 cores 
## Testing With Projection Kernel
## Testing With Gaussian Kernel 1
## Testing With Gaussian Kernel 2
## Testing With Gaussian Kernel 3
## Testing With Gaussian Kernel 4
## Testing With Gaussian Kernel 5
## Testing With Cosine Kernel 1
## Testing With Cosine Kernel 2
## Testing With Cosine Kernel 3
## Testing With Cosine Kernel 4
## Testing With Cosine Kernel 5


Warning message in FUN(newX[, i], ...):
“There are p-values that are exactly 1!”


In [23]:
data$var$gene_name <- rownames(data$var)

In [24]:
library(tidyverse)
df <- sparkX3$res_mtest %>% arrange(adjustedPval,combinedPval)
df$gene_name <- rownames(df)
df <- df %>% left_join(data$var, by="gene_name")
df %>% write_csv("/home/bingxing2/ailab/group/ai4bio/sunjianle/mouse_pmc/svg_atac_gas.csv")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter()   masks stats::filter()
✖ dplyr::lag()      masks stats::lag()
✖ readr::read_csv() masks anndata::read_csv()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
